# Diputrax

Analisis de patrones de reclutamiento para comites legislativos de la camra de diputados del Congreso de la Union de los Estados Unidos Mexicanos.

# 1. Resumen ejecutivo

resumen

# 2. Contexto y definicion del area de interes

Contexto


Area de interes
Patrones de reclutamiento de comisiones

# 

# 3. Objetivos y criterios de exito
Objetivo primario
d

Objetivos secundarios
d

Directivas de la evaluacion de modelos
d



#4. Datos

# 4. Datos disponibles (legisdatamxsil)

Los datos provienen del Sistema de Información Legislativa (SIL) de la Secretaría de Gobernación de México ([sil.gobernacion.gob.mx](https://sil.gobernacion.gob.mx)). Se obtuvieron mediante la implementacion de un web scraper que descarga los perfiles públicos de los diputados federales de la Cámara de Diputados del H. Congreso de la Unión, abarcando las legislaturas LVII a LXVI (1997–presente), con aproximadamente 500 diputados por legislatura.

Cada corrida del scraper produce un archivo CSV por legislatura. Cada archivo contiene una fila por perfil descargado y 32 columnas que recogen tanto los datos personales planos como las secciones de trayectoria y comisiones, serializadas como listas JSON dentro de una celda.

La unidad de análisis es el legislador dentro de una legislatura. Un mismo diputado puede aparecer en más de un archivo si fue reelecto en legislaturas distintas. El campo `diputado_id` permite vincular registros del mismo individuo entre legislaturas (se calcula como SHA-256[:12] sobre nombre normalizado y fecha de nacimiento).

---

## Columnas del CSV crudo (extracto del web scraper)

### Identificadores

| Campo | Tipo | Descripción |
|---|---|---|
| **diputado_id** | str | ID estable entre legislaturas: SHA-256[:12] calculado sobre nombre normalizado + fecha de nacimiento. Vacío si faltan nombre o nacimiento. |
| **referencia** | int | ID numérico del perfil en el SIL. Identifica de forma única a un legislador dentro de una legislatura, pero puede cambiar entre legislaturas. |
| **legislatura_num** | int | Número entero de la legislatura (57–66). Asignado por el scraper según la legislatura objetivo. |
| **profile_url** | str | URL directa al perfil del legislador en el SIL. Útil para diagnóstico y verificación manual. |

### Datos personales (tabla principal del perfil)

| Campo | Tipo | Descripción |
|---|---|---|
| **nombre** | str | Nombre completo del legislador extraído del encabezado del perfil. Los prefijos de cargo (Diputado/a, Sen., Lic., etc.) son eliminados automáticamente. |
| **numero_de_la_legislatura** | str | Número de la legislatura tal como aparece en el perfil (p. ej. "LXVI Legislatura"). |
| **periodo_de_la_legislatura** | str | Años del período legislativo (p. ej. "2024-2027"). |
| **partido** | str | Nombre o siglas del partido político al que pertenece el legislador según el SIL. |
| **nacimiento** | str | Fecha de nacimiento en formato DD/MM/YYYY. Puede estar vacía en perfiles incompletos. |
| **entidad** | str | Entidad federativa que representa o por la que fue electo. |
| **ciudad** | str | Ciudad de nacimiento o de referencia del legislador, según disponibilidad en el perfil. |
| **principio_de_eleccion** | str | Tipo de elección: mayoría relativa (MR) o representación proporcional (RP). |
| **ubicacion** | str | Distrito o circunscripción electoral. |
| **correo_electronico** | str | Correo institucional publicado en el perfil. Frecuentemente vacío o desactualizado. |
| **telefono** | str | Teléfono institucional publicado en el perfil. Frecuentemente vacío. |
| **suplente** | str | Nombre completo del suplente registrado. Vacío si no tiene suplente. |
| **suplente_referencia** | str | ID del perfil del suplente en el SIL. Vacío si no tiene suplente. |
| **ultimo_grado_de_estudios** | str | Nivel de escolaridad declarado (p. ej. "Licenciatura", "Maestría", "Doctorado"). Texto libre; varía en ortografía entre perfiles. |
| **preparacion_academica** | str | Campo de formación o carrera declarada. Texto libre (p. ej. "Derecho", "Ingeniería Civil"). |
| **experiencia_legislativa** | str | Texto libre que describe cargos legislativos previos. Puede incluir referencias a diputaciones locales, federales o senadurías anteriores. |
| **redes_sociales** | str | Cuentas de redes sociales publicadas en el perfil. Frecuentemente vacías o desactualizadas. |

### Secciones anidadas (serializadas como JSON)

Cada una de estas columnas contiene una lista de diccionarios serializada como cadena JSON. Los registros vacíos se guardan como `"[]"`. La mayoría de las secciones de trayectoria tienen la estructura `[{"Del año": "...", "Al año": "...", "Experiencia": "..."}, ...]`.

| Campo | Estructura JSON | Descripción |
|---|---|---|
| **comisiones** | `[{Comisión, Puesto, Fecha Inicial, Fecha Final, Estatus}]` | Membresías en comisiones legislativas (ordinarias, especiales, de investigación). Incluye el puesto (Integrante, Secretario, Presidente, etc.) y estatus (activo/concluido). |
| **licencias_reincorporaciones** | `[{Del año, Al año, Experiencia}]` | Registro de licencias tomadas y reincorporaciones al cargo durante la legislatura. |
| **trayectoria_administrativa** | `[{Del año, Al año, Experiencia}]` | Cargos en la administración pública (federal, estatal o municipal), organizaciones civiles, sindicatos, partidos, etc. |
| **trayectoria_legislativa** | `[{Del año, Al año, Experiencia}]` | Cargos legislativos previos (diputaciones locales o federales anteriores, senadurías). |
| **trayectoria_politica** | `[{Del año, Al año, Experiencia}]` | Participación en partidos políticos, candidaturas, militancia y cargos internos partidistas. |
| **trayectoria_academica** | `[{Del año, Al año, Experiencia}]` | Estudios realizados: institución, grado y período. Puede incluir cursos, diplomados y formación en el extranjero. |
| **trayectoria_empresarial** | `[{Del año, Al año, Experiencia}]` | Actividad en el sector privado o iniciativa privada: cargos directivos, empresas propias, consultoría. |
| **otros_rubros** | `[{Del año, Al año, Experiencia}]` | Actividades no clasificadas en las demás categorías. En la legislatura LXVI contiene además la subsección de investigación y docencia antes de que el raspador la separe. |
| **organos_de_gobierno** | `[{...}]` | Participación en órganos de gobierno de organismos autónomos, fideicomisos u otras instituciones. Estructura variable según el perfil. |
| **observaciones** | `[{...}]` | Notas adicionales del perfil. Estructura variable; frecuentemente vacío. |

### Diagnóstico

| Campo | Tipo | Descripción |
|---|---|---|
| **error** | str | Vacío si el perfil fue descargado y procesado sin incidencias. Contiene el mensaje de error (`fetch_failed`, etc.) si la descarga falló. Los perfiles con error tienen el resto de sus campos vacíos. |

---

## Parametros conocidos de datos

**Conocido**: fuente de datos (SIL, público), cobertura temporal (LVII–LXVI, 1997–presente), unidad de análisis (legislador × legislatura), estructura de cada columna, mecanismo de reanudación automática ante interrupciones, método de generación del `diputado_id`.

**Desconocido / evidencia insuficiente**: tasa de perfiles con error por legislatura; completitud real de los campos de texto libre (`preparacion_academica`, `experiencia_legislativa`) dado que son declarativos y su llenado depende de cada legislador; posibles duplicados dentro de una misma legislatura si un legislador aparece en más de un partido (el raspador deduplica por referencia, pero el SIL podría asignar referencias distintas al mismo individuo); cobertura real de `licencias_reincorporaciones` en legislaturas anteriores a LXII; grado de estandarización de los campos de texto libre entre legislaturas.

## Tratamiento de datos diputrax

### Datos explotados para modelo (Diccionario de Datos — Diputados Federales)

Los datos extraidos mediante el web scraper son tratados con el fin de generar una fuenta de datos integrada y optimizada para el analisis de modelos de machine learning. Como resultado de la segunda capa de tratamiento de datos el modelo cuenta con 78 variables registradas por diputado, organizadas en ocho grupos temáticos según el diccionario siguiente.

La unidad de análisis es el diputado en una legislatura determinada. El tratamiento de segundo nivel no garantiza unicidad: un mismo individuo puede aparecer en múltiples legislaturas con un registro distinto por periodo. Se asume que cada entrada corresponde a la participación de un diputado en una legislatura específica.

Dentro del proyecto diputrax se configuro un pipeline (`run_pipeline.py`) que consolida diez archivos CSV de origen (uno por legislatura) en un único archivo Parquet limpio ubicado en `data/database/clean/`, y genera sobre él un reporte de análisis exploratorio en `reports/eda/`. El pipeline añade además dos variables derivadas: `partido_mayoria` y `es_partido_mayoria`.

---

## Grupo 1 — Identificadores

| Campo | Descripción |
|---|---|
| **diputado_id** | Hash único del diputado. |
| **referencia** | ID numérico en la fuente original. |
| **nombre** | Nombre completo del diputado. |
| **legislatura_nombre** | Nombre romano de la legislatura (ej. LIX). |
| **legislatura_num** | Número ordinal de la legislatura (57–66). |
| **partido_nombre** | Nombre completo del partido político. |
| **partido** | Siglas del partido. |
| **partido_mayoria** | Partido con pluralidad de escaños en esa legislatura *(derivada por el pipeline)*. |
| **es_partido_mayoria** | Binario: 1 si el diputado pertenece al partido mayoritario de su legislatura *(variable objetivo principal)*. |
| **source_file** | Archivo CSV de origen. |

---

## Grupo 2 — Datos Personales

| Campo | Descripción |
|---|---|
| **y_nacimiento** | Año de nacimiento. |
| **edad_al_tomar_cargo** | Edad al inicio del mandato legislativo. |
| **grado_estudios_ord** | Nivel educativo ordinal (0 = sin estudios formales, 7 = doctorado). |
| **area_formacion** | Área disciplinaria de estudios (ej. Derecho, Economía, Medicina). |
| **en_licencia** | `True` si estuvo en licencia durante el periodo. |
| **suplente_referencia** | ID de referencia del diputado suplente. |
| **tiene_suplente** | 1 si tiene suplente registrado. |
| **mayoria_relativa** | 1 si fue electo por mayoría relativa; 0 si por representación proporcional. |
| **entidad_codigo** | Código de 3 letras del estado (ej. CDMX, JAL, NL). |
| **distrito_circ** | Número y nombre del distrito uninominal o circunscripción plurinominal. |

---

## Grupo 3 — Comisiones

| Campo | Descripción |
|---|---|
| **n_comisiones** | Total de comisiones ordinarias a las que perteneció. |
| **n_comisiones_especiales** | Total de comisiones especiales. |
| **n_presidencias** | Número de presidencias de comisión ejercidas. |
| **n_secretarias** | Número de secretarías de comisión ejercidas. |
| **presidente_comision** | 1 si presidió al menos una comisión. |
| **lider_comision** | 1 si fue presidente o secretario de alguna comisión. |
| **n_comisiones_nodales** | Comisiones de alta influencia legislativa (ej. Presupuesto, Hacienda). |
| **n_comisiones_tematicas** | Comisiones de política sectorial (ej. Salud, Educación). |
| **n_comisiones_lastre** | Comisiones de bajo perfil político. |

---

## Grupo 4 — Trayectoria Legislativa

| Campo | Descripción |
|---|---|
| **fue_diputado_local** | 1 si fue diputado local antes del cargo federal. |
| **fue_diputado_federal** | 1 si fue diputado federal en alguna legislatura previa. |
| **fue_senador** | 1 si fue senador de la República. |
| **n_cargos_legislativos_prev** | Suma de cargos legislativos previos (diputado local + federal + senador). |
| **n_trayectoria_legislativa** | Conteo ponderado de experiencia legislativa acumulada. |

---

## Grupo 5 — Trayectoria Administrativa

| Campo | Descripción |
|---|---|
| **n_trayectoria_admin** | Conteo total de cargos administrativos desempeñados. |
| **fue_presidente_mun** | 1 si fue presidente municipal. |
| **fue_presidente_org** | 1 si presidió algún organismo público o privado. |
| **fue_director_general** | 1 si fue director general. |
| **fue_secretario_cargo** | 1 si fue secretario de despacho (federal o estatal). |
| **fue_subsecretario** | 1 si fue subsecretario. |
| **fue_director** | 1 si fue director de área. |
| **fue_coordinador** | 1 si fue coordinador. |
| **fue_delegado** | 1 si fue delegado. |
| **fue_asesor** | 1 si fue asesor. |
| **fue_regidor** | 1 si fue regidor en cabildo municipal. |
| **fue_sindico** | 1 si fue síndico municipal. |
| **admin_en_partido** | 1 si tuvo cargo administrativo dentro de un partido político. |
| **admin_en_sindicato** | 1 si tuvo cargo en un sindicato. |
| **admin_en_universidad** | 1 si tuvo cargo en una institución universitaria. |
| **admin_en_gobierno_fed** | 1 si tuvo cargo en el gobierno federal. |
| **admin_en_gobierno_est** | 1 si tuvo cargo en un gobierno estatal. |
| **admin_en_gobierno_mun** | 1 si tuvo cargo en un gobierno municipal. |
| **nivel_cargo_max** | Nivel máximo de cargo administrativo alcanzado (0 = ninguno, 5 = secretario de estado). |

---

## Grupo 6 — Trayectoria Juvenil

| Campo | Descripción |
|---|---|
| **tiene_exp_juvenil** | 1 si tiene experiencia documentada en organizaciones juveniles. |
| **lider_juvenil_partido** | 1 si fue líder juvenil de partido. |
| **lider_juvenil_gobierno** | 1 si tuvo cargo juvenil en gobierno. |
| **miembro_org_juvenil** | 1 si fue miembro de una organización juvenil. |
| **nivel_liderazgo_juvenil** | Nivel de liderazgo juvenil (0–3). |

---

## Grupo 7 — Formación Académica

| Campo | Descripción |
|---|---|
| **tiene_posgrado** | 1 si tiene estudios de posgrado (maestría o superior). |
| **tiene_doctorado** | 1 si tiene doctorado. |
| **estudios_en_extranjero** | 1 si realizó estudios en el extranjero. |
| **univ_publica** | 1 si estudió en universidad pública. |
| **univ_privada** | 1 si estudió en universidad privada. |
| **univ_extranjera** | 1 si estudió en universidad extranjera. |
| **acad_unam** | 1 si estudió en la UNAM. |
| **acad_itesm** | 1 si estudió en el ITESM (Tec de Monterrey). |
| **acad_itam** | 1 si estudió en el ITAM. |
| **acad_ibero** | 1 si estudió en la Universidad Iberoamericana. |
| **acad_udg** | 1 si estudió en la Universidad de Guadalajara. |
| **acad_ipn** | 1 si estudió en el IPN. |
| **acad_uam** | 1 si estudió en la UAM. |
| **acad_anahuac** | 1 si estudió en la Universidad Anáhuac. |
| **acad_uanl** | 1 si estudió en la UANL. |
| **acad_uv** | 1 si estudió en la Universidad Veracruzana. |

---

## Grupo 8 — Conteos de Trayectoria

| Campo | Descripción |
|---|---|
| **n_trayectoria_politica** | Conteo general de cargos políticos (partido, gobierno, legislativo). |
| **n_trayectoria_empresarial** | Conteo de cargos en el sector privado o empresarial. |
| **n_investigacion_docencia** | Conteo de actividades académicas o de investigación. |
| **n_organos_gobierno** | Conteo de participaciones en órganos de gobierno colegiados. |

---

**Conocidos e incógnitas**:

**Conocido**: definición de la variable objetivo, lista de características, legislaturas cubiertas (LVII–LXVI), fuente institucional (CAMARA), lógica de construcción del dataset.

**Desconocido / evidencia insuficiente**: criterio de inclusión de registros (si todos los 500 diputados por legislatura tienen un registro exhaustivo o si para alguna categorias existe sesgo de selección), tasa de cobertura real de variables biográficas, posible doble conteo de diputados que reingresan en múltiples legislaturas, homogeneidad del proceso de codificación a lo largo del tiempo, y representatividad temporal ante cambios institucionales (reforma política 2014, nueva correlación de fuerzas 2018).



# 5. Criterios de interpretabilidad del modelo

Se considera que 

# 6. Alcance del proyecto

dfdfdf

# 7. Fuera de alcance del proyecto

sdfsdf

# 8. Estrategia metodologica

## 8.1 Analisis de los datos y 